In [37]:
import pandas as pd
import random
import csv
from tqdm import tqdm
import string

from collections import Counter

import spacy
# spacy.cli.download("nl_core_news_lg")
nlp = spacy.load("nl_core_news_lg") 

from airouter import AiRouter

client = AiRouter(
   api_key="sk-eXC2ZNKrhHs-9T2Ei1LjOA",
)

In [3]:
path_name = '/Users/sabijn/Documents/PhD/Datasets/chisor_dataset_all/ChiSCor_CoNLL_paper/csv/ChiSCor_master_df_password/ChiSCor_master_df.csv'
df = pd.read_csv(path_name, index_col=0)

In [39]:
pos_tags = []
pos_tag_story_begin = []
for story in df['story_raw']:
    doc = nlp(story)

    sent_list = list(doc.sents)

    for i, sent in enumerate(sent_list):
        first_word = sent[0]
        if i == 0:
            pos_tag_story_begin.append(first_word.pos_)

        pos_tags.append(first_word.pos_)

In [45]:
pos_counter = Counter(pos_tag_story_begin)

In [8]:
nouns = set()
adjectives = set()
verbs = set()

for story in df['story_raw']:
    for token in nlp(story):
        if token.pos_ == 'NOUN':
            nouns.add(token.lemma_)
        elif token.pos_ == 'VERB':
            verbs.add(token.lemma_)
        elif token.pos_ == 'ADJ':
            adjectives.add(token.lemma_)

In [9]:
nouns, adjectives, verbs = list(nouns), list(adjectives), list(verbs)

In [43]:
def select_pos_tag(weighted=True):
    if weighted:
        tags = list(pos_counter.keys())
        frequencies = list(pos_counter.values())
        sample = random.choices(tags, weights=frequencies, k=1)[0]
    else:
        sample = pos_tags[random.randint(0, len(pos_tags) - 1)]
    
    return sample

In [ ]:
story_features = ['dialoog', 'slecht einde', 'plot twist', 'voorspelling', 'conflict']

In [41]:
def generate_prompt():
    chosen_noun = nouns[random.randint(0, len(nouns) - 1)]
    chosen_adjective = adjectives[random.randint(0, len(adjectives) - 1)]
    chosen_verb = verbs[random.randint(0, len(verbs) - 1)]
    chosen_pos_tag = select_pos_tag()
    chosen_letter = random.choice(string.ascii_lowercase)
    idx_features = [random.randint(0, len(story_features) - 1) for _ in range(2)]

    chosen_features = [story_features[i] for i in idx_features]

    prompt = f"""Vertel een verhaal. Het verhaal moet het volgende werkwoord bevatten: {chosen_verb}, het volgende zelfstandig naamwoord {chosen_noun} en het volgende bijvoegelijk naamwoord {chosen_adjective}.
Begin het verhaal met een {chosen_pos_tag} dat begint met de letter {chosen_letter}."""

    return prompt

In [ ]:
for _ in tqdm(range(1)):
    prompt = generate_prompt()
    print("prompt:", prompt)
    response = client.chat.completions.create(
        messages=[
            {"role": "system", "content": f"""
                Je bent een auteur van een kort verhaal (100-600 woorden).
                Je bent een kind tussen de 4 en 12 en je vertelt een verhaal aan een leeftijdsgenoot. Gebruik woorden en taalconstructies die kinderen van die leeftijd gebruiken.
                Jonge kinderen maken bijvoorbeeld veel gebruik van de constructie "en toen...". Ook gebruiken kinderen vaker dan volwassen de voltooid tegenwoordige tijd, voltooid verleden tijd en verleden tijd.
                Geef het verhaal geen titel of introductie. Het verhaal hoeft geen ego-narratie te zijn, mensen gebruiken een verhaal zelden om hun eigen perspectief te vertellen. Het mag dus vertelt worden
                vanuit het perspectief van iemand anders.
                """},
            {"role": "user", "content": prompt},
        ],
        models=["llama-3.1-8b"]
    )

    completion = response.choices[0].message.content.strip()
    print(completion)

  0%|          | 0/1 [00:00<?, ?it/s]

prompt: Vertel een verhaal. Het verhaal moet het volgende werkwoord bevatten: drijfde, het volgende zelfstandig naamwoord kattenstaart en het volgende bijvoegelijk naamwoord langzamer.
Begin het verhaal met een PRON dat begint met de letter l.


100%|██████████| 1/1 [00:00<00:00,  1.80it/s]

Zij zat op een kano, op het meer. Hij was een vriend van mijn vader. Hij was heel oud. We gingen vaak met hem op het meer. Hij drijfde met zijn hand op het water, alsof hij zo een vis ving.

Hij had ook een hond. Dat dieren had een langzamer lopen dan onze hond. Hij heette Lobo.

Toen wij in de kano waren gingen we naar de overkant. Daar was een plek waar hij nooit kon zitten, vanwege zijn kattenstaart. Zij was heel lang en heel breed.

We gingen naar een klaptoon, die lag op een stuk droog gebied. Daar konden we uitkleden en zwemmen. Hij kwam ook naar de klaptoon. Hij kocht me een ijs. Daar viel ik zo van over. Hij lachte. "Hoe lekker", zei hij.

Terug in de kano liepen we heel langzaam terug. Hij zei: "We gaan nog een keer naar de plek met de klaptoon."


In [13]:
# Open the CSV once in append mode
# Add model=[...] for hardcoding a model
with open("/Users/sabijn/Documents/PhD/code/storylm_p1_data/results/prompt_ChiSCor_like_V3.csv", "a", newline="", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    
    # Write header only once if file is empty
    csvfile.seek(0, 2)  # move to end
    if csvfile.tell() == 0:  
        writer.writerow(["model", "prompt", "completion"])

    for _ in tqdm(range(650)):
        prompt = generate_prompt()
        response = client.chat.completions.create(
            messages=[
                {"role": "user", "content": f"{prompt}"},
            ],
            weighting={
                "latency": 0.0,
            },
            models=["mistral-small"]
        )

        completion = response.choices[0].message.content.strip()

        # Write each row immediately
        writer.writerow([response.model, prompt, completion])


  0%|          | 0/650 [00:04<?, ?it/s]


KeyboardInterrupt: 